# Skip below to see Claude rate workflow bundles.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time


!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.5 MB/s eta 0:00:00
time: 378 µs (started: 2026-05-04 08:34:54 +00:00)


In [3]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "2_Bundle Refinement"


Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 69 (delta 8), reused 43 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 40.59 KiB | 20.29 MiB/s, done.
Resolving deltas: 100% (8/8), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 8.75 KiB | 8.75 MiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 82 (delta 7), reused 81 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 46.13 MiB | 15.11 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (83/83), done.
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects:

In [ ]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

time: 108 ms (started: 2026-05-03 11:27:46 +00:00)


In [ ]:
def flag_guys(flags):

    l = len(flags[0])

    first_flags = flags[0]
    last_flags = flags[-1]

    regular_flags = [True for _ in range(l)]
    for i in range(l):
        if first_flags[i] == False:
            regular_flags[i] = False
        if last_flags[i] == True:
            regular_flags[i] = False


    empty = []

    for i in range(l):
        if regular_flags[i] == True:
            empty.append(i)


    return empty



time: 1.39 ms (started: 2026-05-03 07:28:10 +00:00)


In [ ]:
def bundles_after_workflow(workflow, cap=True):

    domains = ["electronic", "clothing", "food"]

    with open(f"/content/LLM4BEAR/2_Bundle Refinement/historic_bundle_refinement/historical_bundle_changes_electronic_{workflow[0]}.pkl", "rb") as f:

        electronic_intents, electronic_bundle_items, electronic_bundle_indices, electronic_scores, electronic_min_scores, electronic_flags = pickle.load(f)

    electronic_mod_indices = flag_guys(electronic_flags)


    with open(f"/content/LLM4BEAR/2_Bundle Refinement/historic_bundle_refinement/historical_bundle_changes_clothing_{workflow[1]}.pkl", "rb") as f:

        clothing_intents, clothing_bundle_items, clothing_bundle_indices, clothing_scores, clothing_min_scores, clothing_flags = pickle.load(f)


    clothing_mod_indices = flag_guys(clothing_flags)

    with open(f"/content/LLM4BEAR/2_Bundle Refinement/historic_bundle_refinement/historical_bundle_changes_food_{workflow[2]}.pkl", "rb") as f:
        food_intents, food_bundle_items, food_bundle_indices, food_scores, food_min_scores, food_flags = pickle.load(f)

    food_mod_indices = flag_guys(food_flags)


    changed_electronic_scores = copy.deepcopy(electronic_scores)
    changed_clothing_scores = copy.deepcopy(clothing_scores)
    changed_food_scores = copy.deepcopy(food_scores)

    if cap:
        for i in range(21):
            for j in range(len(electronic_bundle_items[0])):
                if len(electronic_bundle_items[i][j]) == 1 or len(electronic_bundle_items[i][j]) == 0:
                    changed_electronic_scores[i][j] = 0

        for i in range(21):
            for j in range(len(clothing_bundle_items[0])):
                if len(clothing_bundle_items[i][j]) == 1 or len(clothing_bundle_items[i][j]) == 0:
                    changed_clothing_scores[i][j] = 0

        for i in range(21):
            for j in range(len(food_bundle_items[0])) or len(food_bundle_items[i][j]) == 0:
                if len(food_bundle_items[i][j]) == 1:
                    changed_food_scores[i][j] = 0
    else:
        for i in range(11):
            for j in range(len(electronic_bundle_items[0])):
                if len(electronic_bundle_items[i][j]) == 1 or len(electronic_bundle_items[i][j]) == 0:
                    changed_electronic_scores[i][j] = 0

        for i in range(11):
            for j in range(len(clothing_bundle_items[0])):
                if len(clothing_bundle_items[i][j]) == 1 or len(clothing_bundle_items[i][j]) == 0:
                    changed_clothing_scores[i][j] = 0

        for i in range(11):
            for j in range(len(food_bundle_items[0])) or len(food_bundle_items[i][j]) == 0:
                if len(food_bundle_items[i][j]) == 1:
                    changed_food_scores[i][j] = 0

    success_electronic = []
    unsuccess_electronic = []

    last_flags_electronic = electronic_flags[-1]
    for i in range(len(last_flags_electronic)):
        if last_flags_electronic[i] == False:
            success_electronic.append(i)
        else:
            unsuccess_electronic.append(i)


    best_index_electronic = []

    for i in unsuccess_electronic:
        score = []
        if cap:
            for j in range(21):
                score.append(changed_electronic_scores[j][i])
            # print(score)
            best_index_electronic.append(score.index(max(score)))
        else:
            for j in range(11):
                score.append(changed_electronic_scores[j][i])
            # print(score)
            best_index_electronic.append(score.index(max(score)))


    pseudo_refined_electronic_bundles = []

    for i in range(len(unsuccess_electronic)):
        pseudo_refined_electronic_bundles.append(electronic_bundle_items[best_index_electronic[i]][unsuccess_electronic[i]])

    pseudo_refined_electronic_intents = []
    for i in range(len(unsuccess_electronic)):
        pseudo_refined_electronic_intents.append(electronic_intents[best_index_electronic[i]][unsuccess_electronic[i]])


    refined_electronic_bundles = []

    for i in range(len(success_electronic)):
        refined_electronic_bundles.append(electronic_bundle_items[-1][success_electronic[i]])

    refined_electronic_intents = []

    for i in range(len(success_electronic)):
        refined_electronic_intents.append(electronic_intents[-1][success_electronic[i]])


    workflow_electronic_bundles = [[] for _ in range(len(success_electronic) + len(unsuccess_electronic))]

    for i in range(len(success_electronic)):
        workflow_electronic_bundles[success_electronic[i]] = refined_electronic_bundles[i]

    for i in range(len(unsuccess_electronic)):
        workflow_electronic_bundles[unsuccess_electronic[i]] = pseudo_refined_electronic_bundles[i]

    workflow_electronic_intents = [[] for _ in range(len(success_electronic) + len(unsuccess_electronic))]

    for i in range(len(success_electronic)):
        workflow_electronic_intents[success_electronic[i]] = refined_electronic_intents[i]

    for i in range(len(unsuccess_electronic)):
        workflow_electronic_intents[unsuccess_electronic[i]] = pseudo_refined_electronic_intents[i]


    success_clothing = []
    unsuccess_clothing = []

    last_flags_clothing = clothing_flags[-1]
    for i in range(len(last_flags_clothing)):
        if last_flags_clothing[i] == False:
            success_clothing.append(i)
        else:
            unsuccess_clothing.append(i)


    best_index_clothing = []

    for i in unsuccess_clothing:
        score = []
        if cap:
            for j in range(21):
                score.append(changed_clothing_scores[j][i])
            # print(score)
            best_index_clothing.append(score.index(max(score)))
        else:
            for j in range(11):
                score.append(changed_clothing_scores[j][i])
            # print(score)
            best_index_clothing.append(score.index(max(score)))


    pseudo_refined_clothing_bundles = []

    for i in range(len(unsuccess_clothing)):
        pseudo_refined_clothing_bundles.append(clothing_bundle_items[best_index_clothing[i]][unsuccess_clothing[i]])

    pseudo_refined_clothing_intents = []
    for i in range(len(unsuccess_clothing)):
        pseudo_refined_clothing_intents.append(clothing_intents[best_index_clothing[i]][unsuccess_clothing[i]])


    refined_clothing_bundles = []

    for i in range(len(success_clothing)):
        refined_clothing_bundles.append(clothing_bundle_items[-1][success_clothing[i]])

    refined_clothing_intents = []

    for i in range(len(success_clothing)):
        refined_clothing_intents.append(clothing_intents[-1][success_clothing[i]])


    workflow_clothing_bundles = [[] for _ in range(len(success_clothing) + len(unsuccess_clothing))]

    for i in range(len(success_clothing)):
        workflow_clothing_bundles[success_clothing[i]] = refined_clothing_bundles[i]

    for i in range(len(unsuccess_clothing)):
        workflow_clothing_bundles[unsuccess_clothing[i]] = pseudo_refined_clothing_bundles[i]

    workflow_clothing_intents = [[] for _ in range(len(success_clothing) + len(unsuccess_clothing))]

    for i in range(len(success_clothing)):
        workflow_clothing_intents[success_clothing[i]] = refined_clothing_intents[i]

    for i in range(len(unsuccess_clothing)):
        workflow_clothing_intents[unsuccess_clothing[i]] = pseudo_refined_clothing_intents[i]


    success_food = []
    unsuccess_food = []

    last_flags_food = food_flags[-1]
    for i in range(len(last_flags_food)):
        if last_flags_food[i] == False:
            success_food.append(i)
        else:
            unsuccess_food.append(i)


    best_index_food = []

    for i in unsuccess_food:
        score = []
        if cap:
            for j in range(21):
                score.append(changed_food_scores[j][i])
            # print(score)
            best_index_food.append(score.index(max(score)))

        else:
            for j in range(11):
                score.append(changed_food_scores[j][i])
            # print(score)
            best_index_food.append(score.index(max(score)))


    pseudo_refined_food_bundles = []

    for i in range(len(unsuccess_food)):
        pseudo_refined_food_bundles.append(food_bundle_items[best_index_food[i]][unsuccess_food[i]])

    pseudo_refined_food_intents = []
    for i in range(len(unsuccess_food)):
        pseudo_refined_food_intents.append(food_intents[best_index_food[i]][unsuccess_food[i]])


    refined_food_bundles = []

    for i in range(len(success_food)):
        refined_food_bundles.append(food_bundle_items[-1][success_food[i]])

    refined_food_intents = []

    for i in range(len(success_food)):
        refined_food_intents.append(food_intents[-1][success_food[i]])


    workflow_food_bundles = [[] for _ in range(len(success_food) + len(unsuccess_food))]

    for i in range(len(success_food)):
        workflow_food_bundles[success_food[i]] = refined_food_bundles[i]

    for i in range(len(unsuccess_food)):
        workflow_food_bundles[unsuccess_food[i]] = pseudo_refined_food_bundles[i]

    workflow_food_intents = [[] for _ in range(len(success_food) + len(unsuccess_food))]

    for i in range(len(success_food)):
        workflow_food_intents[success_food[i]] = refined_food_intents[i]

    for i in range(len(unsuccess_food)):
        workflow_food_intents[unsuccess_food[i]] = pseudo_refined_food_intents[i]

    return workflow_electronic_bundles, workflow_electronic_intents, workflow_clothing_bundles, workflow_clothing_intents, workflow_food_bundles, workflow_food_intents
#

time: 8.39 ms (started: 2026-05-03 07:29:33 +00:00)


In [ ]:

runs = [
    "complete_electronic_session_only_run",
    "complete_clothing_session_only_run",
    "complete_food_session_only_run",
    "complete_electronic_expanded_only_run",
    "complete_clothing_expanded_only_run",
    "complete_food_expanded_only_run",
    "complete_electronic_no_enrichment_run",
    "complete_clothing_no_enrichment_run",
    "complete_food_no_enrichment_run",
    "complete_electronic_no_evaluator_help_run",
    "complete_clothing_no_evaluator_help_run",
    "complete_food_no_evaluator_help_run",
    "complete_electronic_no_graph_help_run", # llm4bear
    "complete_clothing_no_graph_help_run", # llm4bear
    "complete_food_no_graph_help_run" # llm4bear
]

session_only_workflow = runs[0:3]
expanded_only_workflow = runs[3:6]
no_enrichment_workflow = runs[6:9]
no_evaluator_help_workflow = runs[9:12]
llm4bear_workflow = runs[12:15]

session_only_electronic_bundles, session_only_electronic_intents, session_only_clothing_bundles, session_only_clothing_intents, session_only_food_bundles, session_only_food_intents = bundles_after_workflow(session_only_workflow, cap=True)
expanded_only_electronic_bundles, expanded_only_electronic_intents, expanded_only_clothing_bundles, expanded_only_clothing_intents, expanded_only_food_bundles, expanded_only_food_intents = bundles_after_workflow(expanded_only_workflow, cap=True)
no_enrichment_electronic_bundles, no_enrichment_electronic_intents, no_enrichment_clothing_bundles, no_enrichment_clothing_intents, no_enrichment_food_bundles, no_enrichment_food_intents = bundles_after_workflow(no_enrichment_workflow, cap=True)
no_evaluator_help_electronic_bundles, no_evaluator_help_electronic_intents, no_evaluator_help_clothing_bundles, no_evaluator_help_clothing_intents, no_evaluator_help_food_bundles, no_evaluator_help_food_intents = bundles_after_workflow(no_evaluator_help_workflow, cap=False)
llm4bear_electronic_bundles, llm4bear_electronic_intents, llm4bear_clothing_bundles, llm4bear_clothing_intents, llm4bear_food_bundles, llm4bear_food_intents = bundles_after_workflow(llm4bear_workflow, cap=True)



time: 2 s (started: 2026-05-03 07:29:38 +00:00)


In [ ]:
len(session_only_electronic_bundles)

1750

time: 11.1 ms (started: 2026-05-03 07:29:54 +00:00)


In [ ]:
# import pickle as pkl

# with open("/content/drive/My Drive/workflow_bundles/session_only_electronic_bundles.pkl", "wb") as f:
#     pickle.dump([session_only_electronic_bundles, session_only_electronic_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/session_only_clothing_bundles.pkl", "wb") as f:
#     pickle.dump([session_only_clothing_bundles, session_only_clothing_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/session_only_food_bundles.pkl", "wb") as f:
#     pickle.dump([session_only_food_bundles, session_only_food_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_electronic_bundles.pkl", "wb") as f:
#     pickle.dump([expanded_only_electronic_bundles, expanded_only_electronic_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_clothing_bundles.pkl", "wb") as f:
#     pickle.dump([expanded_only_clothing_bundles, expanded_only_clothing_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_food_bundles.pkl", "wb") as f:
#     pickle.dump([expanded_only_food_bundles, expanded_only_food_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_electronic_bundles.pkl", "wb") as f:
#     pickle.dump([no_enrichment_electronic_bundles, no_enrichment_electronic_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_clothing_bundles.pkl", "wb") as f:
#     pickle.dump([no_enrichment_clothing_bundles, no_enrichment_clothing_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_food_bundles.pkl", "wb") as f:
#     pickle.dump([no_enrichment_food_bundles, no_enrichment_food_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_electronic_bundles.pkl", "wb") as f:
#     pickle.dump([no_evaluator_help_electronic_bundles, no_evaluator_help_electronic_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_clothing_bundles.pkl", "wb") as f:
#     pickle.dump([no_evaluator_help_clothing_bundles, no_evaluator_help_clothing_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_food_bundles.pkl", "wb") as f:
#     pickle.dump([no_evaluator_help_food_bundles, no_evaluator_help_food_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_electronic_bundles.pkl", "wb") as f:
#     pickle.dump([llm4bear_electronic_bundles, llm4bear_electronic_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_clothing_bundles.pkl", "wb") as f:
#     pickle.dump([llm4bear_clothing_bundles, llm4bear_clothing_intents], f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_food_bundles.pkl", "wb") as f:
#     pickle.dump([llm4bear_food_bundles, llm4bear_food_intents], f)



time: 387 ms (started: 2026-05-03 07:34:37 +00:00)


# Can start here for the Claude Ratings of workflow bundles

In [ ]:
import pickle as pkl

# with open("/content/drive/My Drive/workflow_bundles/session_only_electronic_bundles.pkl", "rb") as f:
#     session_only_electronic_bundles, session_only_electronic_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/session_only_clothing_bundles.pkl", "rb") as f:
#     session_only_clothing_bundles, session_only_clothing_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/session_only_food_bundles.pkl", "rb") as f:
#     session_only_food_bundles, session_only_food_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_electronic_bundles.pkl", "rb") as f:
#     expanded_only_electronic_bundles, expanded_only_electronic_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_clothing_bundles.pkl", "rb") as f:
#     expanded_only_clothing_bundles, expanded_only_clothing_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/expanded_only_food_bundles.pkl", "rb") as f:
#     expanded_only_food_bundles, expanded_only_food_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_electronic_bundles.pkl", "rb") as f:
#     no_enrichment_electronic_bundles, no_enrichment_electronic_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_clothing_bundles.pkl", "rb") as f:
#     no_enrichment_clothing_bundles, no_enrichment_clothing_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_enrichment_food_bundles.pkl", "rb") as f:
#     no_enrichment_food_bundles, no_enrichment_food_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_electronic_bundles.pkl", "rb") as f:
#     no_evaluator_help_electronic_bundles, no_evaluator_help_electronic_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_clothing_bundles.pkl", "rb") as f:
#     no_evaluator_help_clothing_bundles, no_evaluator_help_clothing_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/no_evaluator_help_food_bundles.pkl", "rb") as f:
#     no_evaluator_help_food_bundles, no_evaluator_help_food_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_electronic_bundles.pkl", "rb") as f:
#     llm4bear_electronic_bundles, llm4bear_electronic_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_clothing_bundles.pkl", "rb") as f:
#     llm4bear_clothing_bundles, llm4bear_clothing_intents = pickle.load(f)

# with open("/content/drive/My Drive/workflow_bundles/llm4bear_food_bundles.pkl", "rb") as f:
#     llm4bear_food_bundles, llm4bear_food_intents = pickle.load(f)



with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/session_only_electronic_bundles.pkl", "rb") as f:
    session_only_electronic_bundles, session_only_electronic_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/session_only_clothing_bundles.pkl", "rb") as f:
    session_only_clothing_bundles, session_only_clothing_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/session_only_food_bundles.pkl", "rb") as f:
    session_only_food_bundles, session_only_food_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/expanded_only_electronic_bundles.pkl", "rb") as f:
    expanded_only_electronic_bundles, expanded_only_electronic_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/expanded_only_clothing_bundles.pkl", "rb") as f:
    expanded_only_clothing_bundles, expanded_only_clothing_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/expanded_only_food_bundles.pkl", "rb") as f:
    expanded_only_food_bundles, expanded_only_food_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_enrichment_electronic_bundles.pkl", "rb") as f:
    no_enrichment_electronic_bundles, no_enrichment_electronic_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_enrichment_clothing_bundles.pkl", "rb") as f:
    no_enrichment_clothing_bundles, no_enrichment_clothing_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_enrichment_food_bundles.pkl", "rb") as f:
    no_enrichment_food_bundles, no_enrichment_food_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_evaluator_help_electronic_bundles.pkl", "rb") as f:
    no_evaluator_help_electronic_bundles, no_evaluator_help_electronic_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_evaluator_help_clothing_bundles.pkl", "rb") as f:
    no_evaluator_help_clothing_bundles, no_evaluator_help_clothing_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/no_evaluator_help_food_bundles.pkl", "rb") as f:
    no_evaluator_help_food_bundles, no_evaluator_help_food_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/llm4bear_electronic_bundles.pkl", "rb") as f:
    llm4bear_electronic_bundles, llm4bear_electronic_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/llm4bear_clothing_bundles.pkl", "rb") as f:
    llm4bear_clothing_bundles, llm4bear_clothing_intents = pickle.load(f)

with open("/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/llm4bear_food_bundles.pkl", "rb") as f:
    llm4bear_food_bundles, llm4bear_food_intents = pickle.load(f)

time: 39.3 ms (started: 2026-05-03 11:27:46 +00:00)


In [ ]:
test_bundle, test_intent = llm4bear_electronic_bundles[200], llm4bear_electronic_intents[200]

time: 727 µs (started: 2026-05-03 07:49:58 +00:00)


In [ ]:
def claude_bundle_prompt(bundle, intent):
    bundle_str = ""
    for i in bundle:
        bundle_str += "-" + i + "\n"
    llm_prompt = f"""Scoring Criteria:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle to purchase.

Bundle intent: {intent}
Bundle Items:
{bundle_str}

**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{{
  "rating: ": float (1-5)
}}
```"""

    return llm_prompt

time: 944 µs (started: 2026-05-03 11:27:46 +00:00)


In [ ]:
print(claude_bundle_prompt(test_bundle, test_intent))

Scoring Criteria:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle to purchase.

Bundle intent: Complete desktop computing setup
Bundle Items:
-HP Pavilion p6-2136b PC Desktop Bundle 20&quot; AMD A6 500GB HDD 6GB DDR3 Desktop PC
-Logitech Wireless Illuminated Keyboard K800
-Dell S2240M CFGKT-IPS-LED 21.5-Inch Screen LED-lit Monitor


**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{
  "rating: ": float (1-5)
}
```
time: 897 µs (started: 2026-05-03 07:56:47 +00:00)


In [ ]:
session_electronic_prompt_list = []
for i in range(1750):
    session_electronic_prompt_list.append(claude_bundle_prompt(session_only_electronic_bundles[i], session_only_electronic_intents[i]))

session_clothing_prompt_list = []
for i in range(1910):
    session_clothing_prompt_list.append(claude_bundle_prompt(session_only_clothing_bundles[i], session_only_clothing_intents[i]))

session_food_prompt_list = []
for i in range(1784):
    session_food_prompt_list.append(claude_bundle_prompt(session_only_food_bundles[i], session_only_food_intents[i]))

expanded_electronic_prompt_list = []
for i in range(1750):
    expanded_electronic_prompt_list.append(claude_bundle_prompt(expanded_only_electronic_bundles[i], expanded_only_electronic_intents[i]))

expanded_clothing_prompt_list = []
for i in range(1910):
    expanded_clothing_prompt_list.append(claude_bundle_prompt(expanded_only_clothing_bundles[i], expanded_only_clothing_intents[i]))

expanded_food_prompt_list = []
for i in range(1784):
    expanded_food_prompt_list.append(claude_bundle_prompt(expanded_only_food_bundles[i], expanded_only_food_intents[i]))

no_enrichment_electronic_prompt_list = []
for i in range(1750):
    no_enrichment_electronic_prompt_list.append(claude_bundle_prompt(no_enrichment_electronic_bundles[i], no_enrichment_electronic_intents[i]))

no_enrichment_clothing_prompt_list = []
for i in range(1910):
    no_enrichment_clothing_prompt_list.append(claude_bundle_prompt(no_enrichment_clothing_bundles[i], no_enrichment_clothing_intents[i]))

no_enrichment_food_prompt_list = []
for i in range(1784):
    no_enrichment_food_prompt_list.append(claude_bundle_prompt(no_enrichment_food_bundles[i], no_enrichment_food_intents[i]))

no_evaluator_electronic_prompt_list = []
for i in range(1750):
    no_evaluator_electronic_prompt_list.append(claude_bundle_prompt(no_evaluator_help_electronic_bundles[i], no_evaluator_help_electronic_intents[i]))

no_evaluator_clothing_prompt_list = []
for i in range(1910):
    no_evaluator_clothing_prompt_list.append(claude_bundle_prompt(no_evaluator_help_clothing_bundles[i], no_evaluator_help_clothing_intents[i]))

no_evaluator_food_prompt_list = []
for i in range(1784):
    no_evaluator_food_prompt_list.append(claude_bundle_prompt(no_evaluator_help_food_bundles[i], no_evaluator_help_food_intents[i]))

llm4bear_electronic_prompt_list = []
for i in range(1750):
    llm4bear_electronic_prompt_list.append(claude_bundle_prompt(llm4bear_electronic_bundles[i], llm4bear_electronic_intents[i]))

llm4bear_clothing_prompt_list = []
for i in range(1910):
    llm4bear_clothing_prompt_list.append(claude_bundle_prompt(llm4bear_clothing_bundles[i], llm4bear_clothing_intents[i]))

llm4bear_food_prompt_list = []
for i in range(1784):
    llm4bear_food_prompt_list.append(claude_bundle_prompt(llm4bear_food_bundles[i], llm4bear_food_intents[i]))

time: 127 ms (started: 2026-05-03 11:05:05 +00:00)


In [ ]:

bundlerec_elec_prompt_list = []
for i in range(1750):
    bundlerec_elec_prompt_list.append(claude_bundle_prompt(electronic_bundles_items[i], electronics_intent['intent'][i]))

bundlerec_clothing_prompt_list = []
for i in range(1910):
    bundlerec_clothing_prompt_list.append(claude_bundle_prompt(clothing_bundles_items[i], clothing_intent['intent'][i]))

bundlerec_food_prompt_list = []
for i in range(1784):
    bundlerec_food_prompt_list.append(claude_bundle_prompt(food_bundles_items[i], food_intent['intent'][i]))

time: 76.3 ms (started: 2026-05-03 11:27:46 +00:00)


In [ ]:
print(bundlerec_elec_prompt_list[1700])

Scoring Criteria:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle to purchase.

Bundle intent: They have a sterio and a frame for the serio then a tv.
Bundle Items:
-Metra 99-5717 Taurus/Sable 04-07 Dash kit
-Kenwood KDC-152 In-Dash MP3/WMA CD Receiver
-Coby LEDTV1526 15&quot; LED High Definition TV


**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{
  "rating: ": float (1-5)
}
```
time: 2.31 ms (started: 2026-05-03 11:01:03 +00:00)


In [ ]:
from bs4 import BeautifulSoup

from google.colab import userdata

open_secret_key = userdata.get('open_router')

if open_secret_key:
  print("OpenRouter Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


from openai import OpenAI


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_secret_key,
)


OpenRouter Token retrieved successfully.
time: 1.44 s (started: 2026-05-03 11:27:46 +00:00)


In [ ]:
from tqdm.asyncio import tqdm_asyncio

async def openrouter_request(user, model_id, system=None):
    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 5)):
        try:
            response = await async_client.chat.completions.create(
                model=model_id, # Now dynamic!
                messages=message,
                temperature=0,
                max_tokens=2000, # Adjust based on bundle length
                # Optional: extra_body is where OpenRouter specific features go
                extra_body={
                    "provider": {"require_parameters": True}
                }
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 200.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error with {model_id}: {e}. Retrying in {round(sleep_dur, 2)}s.")
            await asyncio.sleep(sleep_dur)

    return None


async def run_experiment(prompts, model_id, system=None, batch_size=20):
    results = []
    print(f"🚀 Initializing experiment for: {model_id}")

    # Process in actual chunks to prevent event loop congestion
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]

        tasks = [
            openrouter_request(d["prompts"], model_id, system=system)
            for d in batch
        ]

        # Gather the current batch
        batch_results = await tqdm_asyncio.gather(
            *tasks,
            desc=f"📊 {model_id.split('/')[-1]} [{i}/{len(prompts)}]",
            leave=False
        )

        results.extend(batch_results)

        # MANDATORY COOL-DOWN: Let the API breathe between batches
        # This prevents the "Request Timed Out" loop
        await asyncio.sleep(5)

    print(f"✅ {model_id} — All {len(results)} requests completed.\n")
    return results

time: 2.95 ms (started: 2026-05-03 11:27:48 +00:00)


In [ ]:
import pickle as pkl

async def openrouter_run_tests(prompt_list, workflow_domain):

    system_message = "You are an expert bundle strategist tasked with distinguishing the quality of a bundle out of 5."

    baseline_models = [
        "anthropic/claude-3-5-haiku",

    ]

    names = ["claude"]

    batch_sizes = [10]

    claude_prompt_list = [{"prompts": p} for p in prompt_list]

    # print("Number of prompts:", len(claude_prompt_list))
    # claude_prompt_list = [{"prompts": prompt_list[0]}]

    print(prompt_list[0])

    for i in range(0,1):
        zero_shot_responses = await run_experiment(claude_prompt_list, model_id=baseline_models[i], system=system_message, batch_size=batch_sizes[i])


        with open(f"/content/drive/My Drive/workflow_bundles/{workflow_domain}_claude_responses.pkl", 'wb') as f:
            pkl.dump(zero_shot_responses, f)

    # print(prompt_list[0])
    print(zero_shot_responses[0])

time: 22.6 ms (started: 2026-05-03 11:27:57 +00:00)


In [ ]:
domain_worknames = [
    "electronic_session_only_run",
    "clothing_session_only_run",
    "food_session_only_run",
    "electronic_expanded_only_run",
    "clothing_expanded_only_run",
    "food_expanded_only_run",
    "electronic_no_enrichment_run",
    "clothing_no_enrichment_run",
    "food_no_enrichment_run",
    "electronic_no_evaluator_help_run",
    "clothing_no_evaluator_help_run",
    "food_no_evaluator_help_run",
    "electronic_llm4bear_run", # llm4bear
    "clothing_llm4bear_run", # llm4bear
    "food_llm4bear_run" # llm4bear
]


# await openrouter_run_tests(session_electronic_prompt_list, domain_worknames[0])
# await openrouter_run_tests(session_clothing_prompt_list, domain_worknames[1])
# await openrouter_run_tests(session_food_prompt_list, domain_worknames[2])
# await openrouter_run_tests(expanded_electronic_prompt_list, domain_worknames[3])
# await openrouter_run_tests(expanded_clothing_prompt_list, domain_worknames[4])
# await openrouter_run_tests(expanded_food_prompt_list, domain_worknames[5])
# await openrouter_run_tests(no_enrichment_electronic_prompt_list, domain_worknames[6])
# await openrouter_run_tests(no_enrichment_clothing_prompt_list, domain_worknames[7])
# await openrouter_run_tests(no_enrichment_food_prompt_list, domain_worknames[8])
# await openrouter_run_tests(no_evaluator_electronic_prompt_list, domain_worknames[9])
# await openrouter_run_tests(no_evaluator_clothing_prompt_list, domain_worknames[10])
# await openrouter_run_tests(no_evaluator_food_prompt_list, domain_worknames[11])
# await openrouter_run_tests(llm4bear_electronic_prompt_list, domain_worknames[12])
# await openrouter_run_tests(llm4bear_clothing_prompt_list, domain_worknames[13])
# await openrouter_run_tests(llm4bear_food_prompt_list, domain_worknames[14])

# await openrouter_run_tests(bundlerec_elec_prompt_list, "electronic_bundlerec_run")
# await openrouter_run_tests(bundlerec_clothing_prompt_list, "clothing_bundlerec_run")
# await openrouter_run_tests(bundlerec_food_prompt_list, "food_bundlerec_run")

Scoring Criteria:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle to purchase.

Bundle intent: Storage Devices
Bundle Items:
-Silicon Power 32GB Firma ZN F80 USB 2.0 Flash Drive, Gray Aluminium (SP032GBUF2F80V1S)
-Transcend 400X - 64 GB Compact Flash Memory Card TS64GCF400 (Blue)
-Toshiba Canvio 750 GB USB 3.0 Basics Portable Hard Drive - HDTB107XK3AA(Black)


**## OUTPUT FORMAT:**
Print your reasoning first. Then, print the separator string `===JSON_START===` on a new line. Finally, provide a single JSON object:

```json
{
  "rating: ": float (1-5)
}
```
🚀 Initializing experiment for: anthropic/claude-3-5-haiku


📊 claude-3-5-haiku [670/1750]:  80%|████████  | 8/10 [00:05<00:00,  2.11it/s]

Error with anthropic/claude-3-5-haiku: 'NoneType' object is not subscriptable. Retrying in 3.51s.


✅ anthropic/claude-3-5-haiku — All 1750 requests completed.

Reasoning:
This bundle focuses on storage devices with different capacities and interfaces, which shows some coherence. The items represent various storage technologies:
- USB Flash Drive (32GB): Portable, low-capacity storage
- Compact Flash Memory Card (64GB): Camera/professional media storage
- Portable Hard Drive (750GB): Larger capacity external storage

Strengths:
- Covers different storage needs and form factors
- Ranges from small (32GB) to larger (750GB) storage capacities
- Includes multiple storage technologies

Weaknesses:
- Different interfaces (USB 2.0, USB 3.0, Compact Flash)
- No clear unified purpose beyond generic storage
- Potential compatibility issues across devices

The bundle has moderate value for someone needing diverse storage options, but lacks a precise target audience. It's not a perfectly curated set, but still offers reasonable utility.

===JSON_START===
{
  "rating": 3.5
}
Scoring Criteria:
1-2

✅ anthropic/claude-3-5-haiku — All 1910 requests completed.

Reasoning:
The bundle consists of two tops from the same brand (PattyBoutik), which shows some consistency. However, the items are quite different in style:
- The first is a cowl neck, backless, chain-draped halter top (likely more dressy/evening wear)
- The second is a knit jumper/tunic with a shawl collar and short sleeves (more casual/daytime wear)

While both are tops, they serve very different purposes and styling needs. The lack of cohesive style or versatility between the pieces suggests a weak bundle connection. The items don't complement each other well for mixing and matching or creating a unified wardrobe look.

The only positive is that they're from the same brand, but this alone isn't enough to make a strong bundle.

===JSON_START===
{
  "rating": 2
}
Scoring Criteria:
1-2: Low-quality: No/weak relations between bundle items.
3: Needs improvement: One/two modifications needed.
4-5: High-quality: Reasonable bundle

✅ anthropic/claude-3-5-haiku — All 1784 requests completed.

Reasoning:
This bundle has some challenges in maintaining a cohesive theme. While it starts with a strong Asian food product focus (Japanese rice seasonings, seaweed snacks, ochazuke), the inclusion of Faygo soda (an American brand) and Moonstruck Chocolate (which seems unrelated) disrupts the Asian food theme. 

Positive aspects:
- Multiple Japanese and Korean food products
- Variety of seaweed and rice-related items
- Consistent with Asian snack/condiment theme

Negative aspects:
- Faygo soda is completely out of context
- Chocolate collection seems unrelated
- Lacks a clear, tight bundling strategy

The bundle would benefit from removing the non-Asian items and potentially adding more complementary Asian food products or snacks. As it stands, the bundle feels somewhat random and lacks a strong, coherent value proposition.

===JSON_START===
{
  "rating": 2.5
}
time: 1h 38min 50s (started: 2026-05-03 11:28:01 +00:00)


In [4]:


def extract_json_simple_replace(response_text):
    """
    Extracts a JSON object from a string that has a "===JSON_START===" separator.

    This function isolates the JSON by finding the first '{' and last '}'
    to ensure it works correctly even with markdown fences or extra whitespace.

    Args:
        response_text (str): The full string containing the separator and JSON.

    Returns:
        dict: The parsed JSON object as a Python dictionary, or None if an error occurs.
    """
    try:
        # 1. Get the text after the separator
        json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries of the JSON object
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        # 3. Slice the string to get only the valid JSON
        # This will fail gracefully in the json.loads() if a brace isn't found
        json_string = json_part[first_brace : last_brace + 1]

        # 4. Parse the clean string
        parsed_json = json.loads(json_string)
        return parsed_json

    except IndexError:
        print("Error: The separator '===JSON_START===' was not found.")
        return None
    except json.JSONDecodeError:
        print("Error: Could not find or parse a valid JSON object after the separator.")
        return None


def rating_retrieval(data):

    if data and "rating" in data:
        return data["rating"]
    else:
        return None

time: 1.43 ms (started: 2026-05-04 08:35:16 +00:00)


In [8]:
domain_worknames = [
    "electronic_session_only_run",
    "clothing_session_only_run",
    "food_session_only_run",
    "electronic_expanded_only_run",
    "clothing_expanded_only_run",
    "food_expanded_only_run",
    "electronic_no_enrichment_run",
    "clothing_no_enrichment_run",
    "food_no_enrichment_run",
    "electronic_no_evaluator_help_run",
    "clothing_no_evaluator_help_run",
    "food_no_evaluator_help_run",
    "electronic_llm4bear_run", # llm4bear
    "clothing_llm4bear_run", # llm4bear
    "food_llm4bear_run" # llm4bear
]

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[0]}_claude_responses.pkl", 'rb') as f:
#     session_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[1]}_claude_responses.pkl", 'rb') as f:
#     session_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[2]}_claude_responses.pkl", 'rb') as f:
#     session_food_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[3]}_claude_responses.pkl", 'rb') as f:
#     expanded_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[4]}_claude_responses.pkl", 'rb') as f:
#     expanded_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[5]}_claude_responses.pkl", 'rb') as f:
#     expanded_food_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[6]}_claude_responses.pkl", 'rb') as f:
#     no_enrichment_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[7]}_claude_responses.pkl", 'rb') as f:
#     no_enrichment_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[8]}_claude_responses.pkl", 'rb') as f:
#     no_enrichment_food_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[9]}_claude_responses.pkl", 'rb') as f:
#     no_evaluator_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[10]}_claude_responses.pkl", 'rb') as f:
#     no_evaluator_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[11]}_claude_responses.pkl", 'rb') as f:
#     no_evaluator_food_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[12]}_claude_responses.pkl", 'rb') as f:
#     llm4bear_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[13]}_claude_responses.pkl", 'rb') as f:
#     llm4bear_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/{domain_worknames[14]}_claude_responses.pkl", 'rb') as f:
#     llm4bear_food_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/electronic_bundlerec_run_claude_responses.pkl", 'rb') as f:
#     bundlerec_elec_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/clothing_bundlerec_run_claude_responses.pkl", 'rb') as f:
#     bundlerec_clothing_responses = pkl.load(f)

# with open(f"/content/drive/My Drive/workflow_bundles/food_bundlerec_run_claude_responses.pkl", 'rb') as f:
#     bundlerec_food_responses = pkl.load(f)


with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[0]}_claude_responses.pkl", 'rb') as f:
    session_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[1]}_claude_responses.pkl", 'rb') as f:
    session_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[2]}_claude_responses.pkl", 'rb') as f:
    session_food_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[3]}_claude_responses.pkl", 'rb') as f:
    expanded_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[4]}_claude_responses.pkl", 'rb') as f:
    expanded_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[5]}_claude_responses.pkl", 'rb') as f:
    expanded_food_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[6]}_claude_responses.pkl", 'rb') as f:
    no_enrichment_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[7]}_claude_responses.pkl", 'rb') as f:
    no_enrichment_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[8]}_claude_responses.pkl", 'rb') as f:
    no_enrichment_food_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[9]}_claude_responses.pkl", 'rb') as f:
    no_evaluator_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[10]}_claude_responses.pkl", 'rb') as f:
    no_evaluator_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[11]}_claude_responses.pkl", 'rb') as f:
    no_evaluator_food_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[12]}_claude_responses.pkl", 'rb') as f:
    llm4bear_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[13]}_claude_responses.pkl", 'rb') as f:
    llm4bear_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_worknames[14]}_claude_responses.pkl", 'rb') as f:
    llm4bear_food_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/electronic_bundlerec_run_claude_responses.pkl", 'rb') as f:
    bundlerec_elec_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/clothing_bundlerec_run_claude_responses.pkl", 'rb') as f:
    bundlerec_clothing_responses = pkl.load(f)

with open(f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/food_bundlerec_run_claude_responses.pkl", 'rb') as f:
    bundlerec_food_responses = pkl.load(f)


time: 57.3 ms (started: 2026-05-04 08:36:31 +00:00)


In [9]:
def process_ratings(response_list, list_name):
    extracted_ratings = []
    for idx, resp in enumerate(response_list):
        try:
            # Attempt extraction
            json_data = extract_json_simple_replace(resp)
            rating = rating_retrieval(json_data)
            extracted_ratings.append(rating)
        except Exception as e:
            # If it fails, print the index and the list it belongs to
            print(f"Error in {list_name} at index {idx}: {e}")
            print(f"Raw Response snippet: {str(resp)[:100]}...") # Print a snippet to help identify it
            extracted_ratings.append(None) # Keep list length consistent

    return extracted_ratings

time: 1.63 ms (started: 2026-05-04 08:36:35 +00:00)


In [10]:
domain_worknames = [
    "electronic_session_only_run",
    "clothing_session_only_run",
    "food_session_only_run",
    "electronic_expanded_only_run",
    "clothing_expanded_only_run",
    "food_expanded_only_run",
    "electronic_no_enrichment_run",
    "clothing_no_enrichment_run",
    "food_no_enrichment_run",
    "electronic_no_evaluator_help_run",
    "clothing_no_evaluator_help_run",
    "food_no_evaluator_help_run",
    "electronic_llm4bear_run", # llm4bear
    "clothing_llm4bear_run", # llm4bear
    "food_llm4bear_run", # llm4bear
    "electronic_bundlerec_run",
    "clothing_bundlerec_run",
    "food_bundlerec_run"
]


time: 658 µs (started: 2026-05-04 08:36:37 +00:00)


In [11]:
# A dictionary to hold your final results
all_ratings_results = {}

# Define the names for your variables if you still want separate lists later
# They follow the exact order of your domain_worknames
list_labels = [
    "session_elec", "session_clothing", "session_food",
    "expanded_elec", "expanded_clothing", "expanded_food",
    "no_enrichment_elec", "no_enrichment_clothing", "no_enrichment_food",
    "no_evaluator_elec", "no_evaluator_clothing", "no_evaluator_food",
    "llm4bear_elec", "llm4bear_clothing", "llm4bear_food",
    "bundlerec_elec", "bundlerec_clothing", "bundlerec_food"
]

for idx, domain_name in enumerate(domain_worknames):
    # file_path = f"/content/drive/My Drive/workflow_bundles/{domain_name}_claude_responses.pkl"
    file_path = f"/content/LLM4BEAR/2_Bundle Refinement/workflow_bundles/{domain_name}_claude_responses.pkl"


    label = list_labels[idx]

    try:
        with open(file_path, 'rb') as f:
            responses = pkl.load(f)

        current_list_ratings = []

        for item_idx, resp in enumerate(responses):
            try:
                # 1. Extract JSON
                json_data = extract_json_simple_replace(resp)
                # 2. Get Rating
                rating = rating_retrieval(json_data)
                current_list_ratings.append(rating)
            except Exception as e:
                # This fulfills your requirement: print index (item_idx) on error
                print(f"--- PARSE ERROR ---")
                print(f"List: {label} (domain_worknames[{idx}])")
                print(f"Index in list: {item_idx}")
                print(f"Reason: {e}")
                print(f"-------------------")
                current_list_ratings.append(None)

        all_ratings_results[label] = current_list_ratings

    except FileNotFoundError:
        print(f"Warning: File not found for {domain_name}")

Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START

In [ ]:
ratings_llm4bear_elec = all_ratings_results.get('llm4bear_elec')

time: 627 µs (started: 2026-05-03 09:17:19 +00:00)


In [ ]:
ratings_llm4bear_elec

[3.5,
 4.5,
 4.5,
 5,
 5,
 4.5,
 4.5,
 5,
 4.5,
 5,
 5,
 4.5,
 3.5,
 5,
 4.5,
 4.5,
 4.0,
 5,
 4.5,
 4.5,
 5,
 4.5,
 4.5,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.0,
 4.0,
 4.5,
 4.5,
 3,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 5,
 5,
 4.0,
 4.5,
 4.5,
 5,
 4.5,
 5,
 5,
 5,
 3,
 4.5,
 4.5,
 4.5,
 4.0,
 4.5,
 5,
 4.5,
 4.5,
 4.5,
 5,
 4.5,
 4.5,
 5,
 4.0,
 4.5,
 5,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.5,
 4.5,
 3,
 4.0,
 5,
 5,
 4.0,
 4.0,
 4.5,
 4.5,
 5,
 4.5,
 3.5,
 4.5,
 4.0,
 4.5,
 4.0,
 2.5,
 4.5,
 4.0,
 5,
 4.5,
 4.5,
 5,
 4.5,
 3,
 4.0,
 3.0,
 5,
 2.5,
 5,
 4,
 4.0,
 4.5,
 5,
 3.5,
 4.0,
 3,
 4.0,
 None,
 4.5,
 4.0,
 4.0,
 4.5,
 5,
 4.5,
 5,
 5,
 5,
 5,
 4.5,
 4.5,
 4.5,
 4.5,
 5,
 5,
 4.5,
 4.5,
 4.5,
 5,
 5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.5,
 4.5,
 4.5,
 4.5,
 4.5,
 3.5,
 4.0,
 4.5,
 5,
 5,
 5,
 4.5,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 5,
 4.5,
 4.0,
 4.0,
 4.5,
 4.5,
 4.5,
 4.5,
 4.5,
 5,
 4.5,
 4.0,
 4.0,
 5,
 4.5,
 5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.0,
 4.5,
 4.5,
 4.5,
 4.5,
 5,
 4.5,
 4.5,

time: 12.4 ms (started: 2026-05-03 09:17:27 +00:00)


In [12]:
# Iterate through every list label you've defined
for label in list_labels:
    # Retrieve the specific ratings list from your results dictionary
    current_ratings = all_ratings_results.get(label)

    if current_ratings is None:
        print(f"--- {label}: List not found ---")
        continue

    # Find indices where the value is None
    bad_indices = [idx for idx, val in enumerate(current_ratings) if val is None]

    # Only print if there are actually missing ratings
    if bad_indices:
        print(f"Missing ratings in [{label}] at indices: {bad_indices}")
    else:
        print(f"[{label}]: All ratings parsed successfully.")

Missing ratings in [session_elec] at indices: [159, 167, 332, 1316, 1480, 1625]
Missing ratings in [session_clothing] at indices: [753, 1282, 1294, 1417, 1879, 1907]
Missing ratings in [session_food] at indices: [92, 477, 688, 783]
Missing ratings in [expanded_elec] at indices: [244, 1162, 1165, 1366]
Missing ratings in [expanded_clothing] at indices: [382, 402, 1273, 1666, 1765]
Missing ratings in [expanded_food] at indices: [234, 430, 474, 513, 525, 960, 996, 1774]
Missing ratings in [no_enrichment_elec] at indices: [25, 371, 478, 711, 720, 824, 1498, 1576]
Missing ratings in [no_enrichment_clothing] at indices: [356, 565, 584, 635, 753, 1453, 1494]
Missing ratings in [no_enrichment_food] at indices: [309, 354, 818, 880, 888, 1344, 1733]
Missing ratings in [no_evaluator_elec] at indices: [137, 675, 1021, 1066, 1085, 1357, 1358, 1646, 1720]
Missing ratings in [no_evaluator_clothing] at indices: [257, 859, 896, 1090, 1139, 1298, 1346, 1488, 1557, 1679, 1793, 1806]
Missing ratings in [n

In [13]:
session_elec_missing = [159, 167, 332, 1316, 1480, 1625]
session_clothing_missing = [753, 1282, 1294, 1417, 1879, 1907]
session_food_missing = [92, 477, 688, 783]

expanded_elec_missing = [244, 1162, 1165, 1366]
expanded_clothing_missing = [382, 402, 1273, 1666, 1765]
expanded_food_missing = [234, 430, 474, 513, 525, 960, 996, 1774]

no_enrichment_elec_missing = [25, 371, 478, 711, 720, 824, 1498, 1576]
no_enrichment_clothing_missing = [356, 565, 584, 635, 753, 1453, 1494]
no_enrichment_food_missing = [309, 354, 818, 880, 888, 1344, 1733]

no_evaluator_elec_missing = [137, 675, 1021, 1066, 1085, 1357, 1358, 1646, 1720]
no_evaluator_clothing_missing = [257, 859, 896, 1090, 1139, 1298, 1346, 1488, 1557, 1679, 1793, 1806]
no_evaluator_food_missing = [800, 824, 1374, 1566, 1568, 1640, 1678, 1700, 1741, 1758, 1772]

llm4bear_elec_missing = [109, 235, 1300, 1408, 1558]
llm4bear_clothing_missing = [29, 382, 857, 1024, 1650]
llm4bear_food_missing = [10, 22, 470, 926, 1200, 1284, 1462, 1593, 1623, 1714]

bundlerec_elec_missing = [162, 257, 258, 273, 571, 773]
bundlerec_clothing_missing = [95, 1083, 1215, 1366, 1502, 1602]
bundlerec_food_missing = [32, 274, 802, 945, 993, 1113, 1264, 1353, 1424, 1652]

time: 1.67 ms (started: 2026-05-04 08:36:45 +00:00)


In [14]:
# Map the missing index lists to the actual response lists
missing_data_map = {
    "session_elec": (session_elec_missing, session_elec_responses),
    "session_clothing": (session_clothing_missing, session_clothing_responses),
    "session_food": (session_food_missing, session_food_responses),
    "expanded_elec": (expanded_elec_missing, expanded_elec_responses),
    "expanded_clothing": (expanded_clothing_missing, expanded_clothing_responses),
    "expanded_food": (expanded_food_missing, expanded_food_responses),
    "no_enrichment_elec": (no_enrichment_elec_missing, no_enrichment_elec_responses),
    "no_enrichment_clothing": (no_enrichment_clothing_missing, no_enrichment_clothing_responses),
    "no_enrichment_food": (no_enrichment_food_missing, no_enrichment_food_responses),
    "no_evaluator_elec": (no_evaluator_elec_missing, no_evaluator_elec_responses),
    "no_evaluator_clothing": (no_evaluator_clothing_missing, no_evaluator_clothing_responses),
    "no_evaluator_food": (no_evaluator_food_missing, no_evaluator_food_responses),
    "llm4bear_elec": (llm4bear_elec_missing, llm4bear_elec_responses),
    "llm4bear_clothing": (llm4bear_clothing_missing, llm4bear_clothing_responses),
    "llm4bear_food": (llm4bear_food_missing, llm4bear_food_responses),
    "bundlerec_elec": (bundlerec_elec_missing, bundlerec_elec_responses),
    "bundlerec_clothing": (bundlerec_clothing_missing, bundlerec_clothing_responses),
    "bundlerec_food": (bundlerec_food_missing, bundlerec_food_responses)
}

for label, (indices, responses) in missing_data_map.items():
    print(f"\n{'='*30}\nCATALOGUE: {label}\n{'='*30}")

    for idx in indices:
        print(f"\n--- [ INDEX: {idx} ] ---")
        try:
            print(responses[idx])
        except IndexError:
            print(f"Error: Index {idx} is out of bounds for this list.")
        print("-" * 25)


CATALOGUE: session_elec

--- [ INDEX: 159 ] ---
Reasoning:
The bundle quality consists of two complementary items specifically designed for the enhance and protect the THE asus Transformer:
- The Mobile Dock provides provides additional functionality like keyboard/extended battery
- The Screen protector provides military-grade protection protection screen defense 
items are compatible, purpose-aligned, and from mutually beneficial. The screen bundle shows strategic pairing to maximize the device utility and longevity...The bundle demonstrates strong inter-item relations, synergy, with with clear protective and and functional benefits...

======JSON_START__START===

{

  rating: rating 4.55
}}
-------------------------

--- [ INDEX: 167 ] ---
Reasoning:
This bundle appears to be a solid a--out package with complementary items::
- The digital camera (( kodEasyC195 )) is the core item
- The memory card (((Samsung 16GB) GB) provides essential storage storage
- The camera case ((Logic DC-B

In [ ]:
# session_elec_missing = [159, 167, 332, 1316, 1480, 1625] = [4.55, 4, 3.5, 4.5, 4.5, 4.5]
# session_clothing_missing = [753, 1282, 1294, 1417, 1879, 1907] = [3, 3.55, 4, 4, 4.05, 4]
# session_food_missing = [92, 477, 688, 783] = [4.2, 3, 4.5, 3]

# expanded_elec_missing = [244, 1162, 1165, 1366] = [3.3, 3.5, 3.5, 4.5]
# expanded_clothing_missing = [382, 402, 1273, 1666, 1765] = [4.35, 4.20, 4.0, 3.0, 3.25]
# expanded_food_missing = [234, 430, 474, 513, 525, 960, 996, 1774] = [4.5, 2, 3.5, 4.35, 4.0, 4.5, 4.0, 4.5]

# no_enrichment_elec_missing = [25, 371, 478, 711, 720, 824, 1498, 1576] = [4.5, 4.55, 4.35, 2.5, 3.5, 3.5, 4.0, 4.0]
# no_enrichment_clothing_missing = [356, 565, 584, 635, 753, 1453, 1494] = [4.2, 3, 3.55, 4, 3, 3, 4.7]
# no_enrichment_food_missing = [309, 354, 818, 880, 888, 1344, 1733] = [4.0, 4.55, 4.5, 4.5, 4.2, 4.5, 3]

# no_evaluator_elec_missing = [137, 675, 1021, 1066, 1085, 1357, 1358, 1646, 1720] = [2, 4.2, 3, 2.5, 2.55, 3.2, 3, 3, 1.5]
# no_evaluator_clothing_missing = [257, 859, 896, 1090, 1139, 1298, 1346, 1488, 1557, 1679, 1793, 1806] = [3.5, 3.0, 4.0, 2, 3, 3, 2, 4, 4.5, 3, 4, 3]
# no_evaluator_food_missing = [800, 824, 1374, 1566, 1568, 1640, 1678, 1700, 1741, 1758, 1772] = [4.25, 4.55, 4, 4.5, 3, 3, 3, 3.5, 4.22, 4.5, 4.5]

# llm4bear_elec_missing = [109, 235, 1300, 1408, 1558] = [4.3, 4, 4.5, 4.5, 4]
# llm4bear_clothing_missing = [29, 382, 857, 1024, 1650] = [4.2, 3, 4.55, 3, 4]
# llm4bear_food_missing = [10, 22, 470, 926, 1200, 1284, 1462, 1593, 1623, 1714] = [4.2, 3, 4.55, 4.5, 3.55, 4.2, 4, 4.2, 4.2, 4.55]

# bundlerec_elec_missing = [162, 257, 258, 273, 571, 773] = [4.5, 4.5, 3.5, 3, 2.5, 4.2]
# bundlerec_clothing_missing = [95, 1083, 1215, 1366, 1502, 1602] = [3.2, 4, 3, 3.55, 4, 3]
# bundlerec_food_missing = [32, 274, 802, 945, 993, 1113, 1264, 1353, 1424, 1652] = [3.5, 4, 3, 4.5, 4.2, 4, 4.55, 4.5, 3.5, 4, 2.3]

In [15]:
ratings_session_elec = []
for i in session_elec_responses:
    ratings_session_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_session_clothing = []
for i in session_clothing_responses:
    ratings_session_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_session_food = []
for i in session_food_responses:
    ratings_session_food.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_expanded_elec = []
for i in expanded_elec_responses:
    ratings_expanded_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_expanded_clothing = []
for i in expanded_clothing_responses:
    ratings_expanded_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_expanded_food = []
for i in expanded_food_responses:
    ratings_expanded_food.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_enrichment_elec = []
for i in no_enrichment_elec_responses:
    ratings_no_enrichment_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_enrichment_clothing = []
for i in no_enrichment_clothing_responses:
    ratings_no_enrichment_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_enrichment_food = []
for i in no_enrichment_food_responses:
    ratings_no_enrichment_food.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_evaluator_elec = []
for i in no_evaluator_elec_responses:
    ratings_no_evaluator_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_evaluator_clothing = []
for i in no_evaluator_clothing_responses:
    ratings_no_evaluator_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_no_evaluator_food = []
for i in no_evaluator_food_responses:
    ratings_no_evaluator_food.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_llm4bear_elec = []
for i in llm4bear_elec_responses:
    ratings_llm4bear_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_llm4bear_clothing = []
for i in llm4bear_clothing_responses:
    ratings_llm4bear_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_llm4bear_food = []
for i in llm4bear_food_responses:
    ratings_llm4bear_food.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_bundlerec_elec = []
for i in bundlerec_elec_responses:
    ratings_bundlerec_elec.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_bundlerec_clothing = []
for i in bundlerec_clothing_responses:
    ratings_bundlerec_clothing.append(rating_retrieval(extract_json_simple_replace(i)))

ratings_bundlerec_food = []
for i in bundlerec_food_responses:
    ratings_bundlerec_food.append(rating_retrieval(extract_json_simple_replace(i)))

Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: Could not find or parse a valid JSON object after the separator.
Error: Could not find or parse a valid JSON object after the separator.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START===' was not found.
Error: The separator '===JSON_START

In [16]:
# A mapping of your ratings lists to the manual correction data you just provided
corrections_map = {
    "session_elec": (ratings_session_elec, session_elec_missing, [4.55, 4, 3.5, 4.5, 4.5, 4.5]),
    "session_clothing": (ratings_session_clothing, session_clothing_missing, [3, 3.55, 4, 4, 4.05, 4]),
    "session_food": (ratings_session_food, session_food_missing, [4.2, 3, 4.5, 3]),

    "expanded_elec": (ratings_expanded_elec, expanded_elec_missing, [3.3, 3.5, 3.5, 4.5]),
    "expanded_clothing": (ratings_expanded_clothing, expanded_clothing_missing, [4.35, 4.20, 4.0, 3.0, 3.25]),
    "expanded_food": (ratings_expanded_food, expanded_food_missing, [4.5, 2, 3.5, 4.35, 4.0, 4.5, 4.0, 4.5]),

    "no_enrichment_elec": (ratings_no_enrichment_elec, no_enrichment_elec_missing, [4.5, 4.55, 4.35, 2.5, 3.5, 3.5, 4.0, 4.0]),
    "no_enrichment_clothing": (ratings_no_enrichment_clothing, no_enrichment_clothing_missing, [4.2, 3, 3.55, 4, 3, 3, 4.7]),
    "no_enrichment_food": (ratings_no_enrichment_food, no_enrichment_food_missing, [4.0, 4.55, 4.5, 4.5, 4.2, 4.5, 3]),

    "no_evaluator_elec": (ratings_no_evaluator_elec, no_evaluator_elec_missing, [2, 4.2, 3, 2.5, 2.55, 3.2, 3, 3, 1.5]),
    "no_evaluator_clothing": (ratings_no_evaluator_clothing, no_evaluator_clothing_missing, [3.5, 3.0, 4.0, 2, 3, 3, 2, 4, 4.5, 3, 4, 3]),
    "no_evaluator_food": (ratings_no_evaluator_food, no_evaluator_food_missing, [4.25, 4.55, 4, 4.5, 3, 3, 3, 3.5, 4.22, 4.5, 4.5]),

    "llm4bear_elec": (ratings_llm4bear_elec, llm4bear_elec_missing, [4.3, 4, 4.5, 4.5, 4]),
    "llm4bear_clothing": (ratings_llm4bear_clothing, llm4bear_clothing_missing, [4.2, 3, 4.55, 3, 4]),
    "llm4bear_food": (ratings_llm4bear_food, llm4bear_food_missing, [4.2, 3, 4.55, 4.5, 3.55, 4.2, 4, 4.2, 4.2, 4.55]),

    "bundlerec_elec": (ratings_bundlerec_elec, bundlerec_elec_missing, [4.5, 4.5, 3.5, 3, 2.5, 4.2]),
    "bundlerec_clothing": (ratings_bundlerec_clothing, bundlerec_clothing_missing, [3.2, 4, 3, 3.55, 4, 3]),
    "bundlerec_food": (ratings_bundlerec_food, bundlerec_food_missing, [3.5, 4, 3, 4.5, 4.2, 4, 4.55, 4.5, 3.5, 4, 2.3]),
}

for label, (target_list, indices, new_values) in corrections_map.items():
    for i, idx in enumerate(indices):
        target_list[idx] = new_values[i]
    print(f"Applied {len(indices)} corrections to {label}.")

Applied 6 corrections to session_elec.
Applied 6 corrections to session_clothing.
Applied 4 corrections to session_food.
Applied 4 corrections to expanded_elec.
Applied 5 corrections to expanded_clothing.
Applied 8 corrections to expanded_food.
Applied 8 corrections to no_enrichment_elec.
Applied 7 corrections to no_enrichment_clothing.
Applied 7 corrections to no_enrichment_food.
Applied 9 corrections to no_evaluator_elec.
Applied 12 corrections to no_evaluator_clothing.
Applied 11 corrections to no_evaluator_food.
Applied 5 corrections to llm4bear_elec.
Applied 5 corrections to llm4bear_clothing.
Applied 10 corrections to llm4bear_food.
Applied 6 corrections to bundlerec_elec.
Applied 6 corrections to bundlerec_clothing.
Applied 10 corrections to bundlerec_food.
time: 6.35 ms (started: 2026-05-04 08:36:55 +00:00)


In [17]:
all_ratings = [
    ratings_session_elec, ratings_session_clothing, ratings_session_food,
    ratings_expanded_elec, ratings_expanded_clothing, ratings_expanded_food,
    ratings_no_enrichment_elec, ratings_no_enrichment_clothing, ratings_no_enrichment_food,
    ratings_no_evaluator_elec, ratings_no_evaluator_clothing, ratings_no_evaluator_food,
    ratings_llm4bear_elec, ratings_llm4bear_clothing, ratings_llm4bear_food,
    ratings_bundlerec_elec, ratings_bundlerec_clothing, ratings_bundlerec_food
]

time: 1.02 ms (started: 2026-05-04 08:36:59 +00:00)


In [ ]:
import numpy as np

# Assuming all_ratings is a list containing your 15 ratings lists in order:
# [ratings_session_elec, ratings_session_clothing, ..., ratings_llm4bear_food]

print(f"{'Mean Rating':<12} | {'Experiment Domain'}")
print("-" * 40)

for i in range(len(all_ratings)):
    # Calculate the mean, ignoring any remaining None values just in case
    # although your manual corrections should have filled them!
    current_mean = np.nanmean([val for val in all_ratings[i] if val is not None])

    # Format the output for readability
    print(f"{current_mean:<12.4f} | {domain_worknames[i]}")

    # Add a newline every 3 domains to separate the experimental groups
    if (i + 1) % 3 == 0:
        print("-" * 40)

Mean Rating  | Experiment Domain
----------------------------------------
3.8837       | electronic_session_only_run
3.6546       | clothing_session_only_run
4.1364       | food_session_only_run
----------------------------------------
4.3907       | electronic_expanded_only_run
3.8540       | clothing_expanded_only_run
4.4233       | food_expanded_only_run
----------------------------------------
4.3961       | electronic_no_enrichment_run
3.9193       | clothing_no_enrichment_run
4.4425       | food_no_enrichment_run
----------------------------------------
4.1678       | electronic_no_evaluator_help_run
3.6644       | clothing_no_evaluator_help_run
4.1195       | food_no_evaluator_help_run
----------------------------------------
4.3796       | electronic_llm4bear_run
3.8510       | clothing_llm4bear_run
4.4318       | food_llm4bear_run
----------------------------------------
3.4351       | electronic_bundlerec_run
3.3642       | clothing_bundlerec_run
3.7261       | food_bundlerec

In [19]:
import numpy as np

# Map the totals you provided to the domains
totals_map = {
    'electronics': 1750,
    'clothing': 1910,
    'food': 1784
}

print(f"{'Mean':<8} | {'>= 4 Count':<10} | {'Hit Rate %':<12} | {'Experiment Domain'}")
print("-" * 75)

for i in range(len(all_ratings)):
    clean_list = [val for val in all_ratings[i] if val is not None]
    current_mean = np.nanmean(clean_list)

    # Count ratings >= 4
    high_count = sum(1 for val in clean_list if val >= 4)

    # Determine which total to use for the percentage calculation
    name_lower = domain_worknames[i].lower()
    if 'elec' in name_lower:
        denom = totals_map['electronics']
    elif 'cloth' in name_lower:
        denom = totals_map['clothing']
    elif 'food' in name_lower:
        denom = totals_map['food']
    else:
        denom = len(clean_list) # Fallback to list length if domain name doesn't match

    hit_rate = (high_count / denom) * 100

    # Print the row with the new column
    print(f"{current_mean:<8.4f} | {high_count:<10} | {hit_rate:<12.2f} | {domain_worknames[i]}")

    if (i + 1) % 3 == 0:
        print("-" * 75)

Mean     | >= 4 Count | Hit Rate %   | Experiment Domain
---------------------------------------------------------------------------
3.8837   | 1175       | 67.14        | electronic_session_only_run
3.6546   | 1090       | 57.07        | clothing_session_only_run
4.1364   | 1423       | 79.76        | food_session_only_run
---------------------------------------------------------------------------
4.3907   | 1542       | 88.11        | electronic_expanded_only_run
3.8540   | 1233       | 64.55        | clothing_expanded_only_run
4.4233   | 1646       | 92.26        | food_expanded_only_run
---------------------------------------------------------------------------
4.3961   | 1540       | 88.00        | electronic_no_enrichment_run
3.9193   | 1309       | 68.53        | clothing_no_enrichment_run
4.4425   | 1651       | 92.54        | food_no_enrichment_run
---------------------------------------------------------------------------
4.1678   | 1382       | 78.97        | electronic_no_e

In [ ]:
# Find indices where the rating is None
bad_indices = [idx for idx, val in enumerate(ratings_llm4bear_elec) if val is None]

print(f"Missing ratings at indices: {bad_indices}")

# To inspect the first error immediately:
if bad_indices:
    first_bad = bad_indices[0]
    print(f"\n--- Inspecting Index {first_bad} ---")
    print(llm4bear_elec_responses[first_bad])

Missing ratings at indices: [109, 235, 1300, 1408, 1558]

--- Inspecting Index 109 ---
Reasoning:
- The bundle contains two networking/networking components
- BUFFALO Awireless Router (Buffalo A
ir) provides network connectivity
- 100FCAT Ethernet cable supports high-/gaming/internet connectivity
-
Components are complementary and support support network infrastructure
-
Appears well-matched for home/or small setup networking needs

Key strengths::
- Direct compatibility 
-/router matching
- Infrastructure support

Reasonable price point point

======JSON_START===_START===

{

  """: 4.3)
}}

The reasoning highlights the complementary nature of the of the components, their their direct infrastructure support support capabilities, and reasonable matching for home/small setup networking needs. The

 4.3rating reflects strong bundling potential with minor optimization room.
time: 3.73 ms (started: 2026-05-03 09:18:20 +00:00)
